In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score,StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import os
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore",category = ConvergenceWarning)
os.chdir('E:/Machine_Learning/Cases/Vehicle Silhouettes/')

In [3]:
vehicle= pd.read_csv("Vehicle.csv")
vehicle

,Comp,Circ,D.Circ,Rad.Ra,Pr.Axis.Ra,Max.L.Ra,Scat.Ra,Elong,Pr.Axis.Rect,Max.L.Rect,Sc.Var.Maxis,Sc.Var.maxis,Ra.Gyr,Skew.Maxis,Skew.maxis,Kurt.maxis,Kurt.Maxis,Holl.Ra,Class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841,93,39,87,183,64,8,169,40,20,134,200,422,149,72,7,25,188,195,saab
842,89,46,84,163,66,11,159,43,20,159,173,368,176,72,1,20,186,197,van
843,106,54,101,222,67,12,222,30,25,173,228,721,200,70,3,4,187,201,saab
844,86,36,78,146,58,7,135,50,18,124,155,270,148,66,0,25,190,195,saab


In [4]:
X,y = vehicle.drop('Class',axis=1),vehicle['Class']
le = LabelEncoder() 
le_y = le.fit_transform(y)
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
gbm = GradientBoostingClassifier(random_state=26)

### Accuracy Scores

accuracy_score :  default value=> cross_val_score will give accurcy_score

In [5]:
result  = cross_val_score(gbm, X, le_y, cv=kfold)
result

array([0.77058824, 0.76923077, 0.78698225, 0.78106509, 0.75147929])

In [6]:
np.mean(result)

np.float64(0.7718691263487644)

Log loss : 

In [7]:
result  = cross_val_score(gbm, X, le_y, cv=kfold, scoring='neg_log_loss')
result

array([-0.46420906, -0.47148921, -0.43516682, -0.47498426, -0.55013841])

### Fine Tuning

In [11]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            gbm = GradientBoostingClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            result = cross_val_score(gbm, X, le_y, cv=kfold, scoring='neg_log_loss')
            scores.append([l,m,n,np.mean(result)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','log_loss'])
df_scores.sort_values(by='log_loss',ascending=False) 

100%|██████████| 10/10 [07:55<00:00, 47.59s/it]


,Learning_rate,Max_depth,n_estimators,log_loss
29,0.207778,2.0,100,-0.461149
52,0.405556,2.0,50,-0.463170
41,0.306667,2.0,100,-0.471728
40,0.306667,2.0,50,-0.471766
31,0.207778,3.0,50,-0.477252
...,...,...,...,...
110,0.900000,NaN,100,-2.932670
108,0.900000,NaN,25,-2.932670
72,0.603333,NaN,25,-2.960540
73,0.603333,NaN,50,-2.960541


#### In above code we have used neg_log_loss, which gives us value of -(log_loss) thats why we sort it in descending order, because we want value nearest to 0

In [8]:
from sklearn.linear_model import LogisticRegression

In [13]:
solvers = ['lbfgs','newton-cg','newton-cholesky']
scores = []
for s in tqdm(solvers):
    lr = LogisticRegression(solver=s)
    result = cross_val_score(lr, X, le_y, cv=kfold, scoring='neg_log_loss')
    scores.append([s,np.mean(result)])
df_scores = pd.DataFrame(scores,columns=['solver','log_loss'])
df_scores.sort_values(by='log_loss',ascending=False)

100%|██████████| 3/3 [00:05<00:00,  1.68s/it]


,solver,log_loss
1,newton-cg,-0.416957
2,newton-cholesky,-0.438304
0,lbfgs,-0.679773


### Including C values

In [14]:
solvers = ['lbfgs','newton-cg','newton-cholesky']
Cs = np.linspace(0.01,3,5)
scores = []
for s in tqdm(solvers):
    for c in Cs:
        lr = LogisticRegression(solver=s,C=c)
        result = cross_val_score(lr, X, le_y, cv=kfold, scoring='neg_log_loss')
        scores.append([s,c,np.mean(result)])
df_scores = pd.DataFrame(scores,columns=['solver','C','log_loss'])
df_scores.sort_values(by='log_loss',ascending=False)

100%|██████████| 3/3 [00:21<00:00,  7.07s/it]


,solver,C,log_loss
6,newton-cg,0.7575,-0.414607
7,newton-cg,1.5050,-0.421581
8,newton-cg,2.2525,-0.427825
9,newton-cg,3.0000,-0.431223
11,newton-cholesky,0.7575,-0.432451
12,newton-cholesky,1.5050,-0.447914
13,newton-cholesky,2.2525,-0.458194
14,newton-cholesky,3.0000,-0.465749
10,newton-cholesky,0.0100,-0.470541
5,newton-cg,0.0100,-0.472532


## Grid Search CV

In [20]:
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=26)

In [21]:
from sklearn.model_selection import GridSearchCV
lr = LogisticRegression()
params = {'solver': ['lbfgs','newton-cg','newton-cholesky'],'C': np.linspace(0.01,3,5)}
gcv = GridSearchCV(lr,param_grid = params ,cv = kfold, scoring = 'neg_log_loss')
gcv.fit(X,le_y)
print("Best param : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best param :  {'C': np.float64(0.7575000000000001), 'solver': 'newton-cg'}
Best score :  -0.41460687009627517


In [23]:
gcv.cv_results_


{'mean_fit_time': array([0.12356515, 0.29057727, 0.02166767, 0.11526937, 0.75233669,
        0.02718468, 0.16275873, 0.63001075, 0.02744288, 0.16276531,
        1.09677205, 0.02766571, 0.15764289, 0.70699253, 0.0287055 ]),
 'std_fit_time': array([0.02070215, 0.05278766, 0.00151456, 0.00590862, 0.49007582,
        0.00156075, 0.00623591, 0.41936352, 0.00194273, 0.01149837,
        0.23255948, 0.00239163, 0.01040994, 0.20387546, 0.00182452]),
 'mean_score_time': array([0.00679994, 0.00390182, 0.00354605, 0.00600691, 0.00405784,
        0.00371413, 0.01073241, 0.00397439, 0.00372148, 0.01012216,
        0.00395293, 0.00350823, 0.00974789, 0.00383029, 0.00523934]),
 'std_score_time': array([0.00254143, 0.00019121, 0.00031827, 0.00225522, 0.00030733,
        0.00020849, 0.00082412, 0.00011941, 0.00014158, 0.00120663,
        0.00012107, 0.00014576, 0.00300788, 0.00014636, 0.00288307]),
 'param_C': masked_array(data=[0.01, 0.01, 0.01, 0.7575000000000001,
                    0.757500000000000